#Detector

## Train an EWC model on tasks 0–8, then save the victim checkpoint before task 9

In [ ]:
CUDA_VISIBLE_DEVICES=0 python -u main_baselines.py \
  --experiment split_cifar100 \
  --approach ewc \
  --lasttask 9 \
  --tasknum 10 \
  --nepochs 20 \
  --batch-size 16 \
  --lamb 500000 \
  --clip 100.0 \
  --lr 0.01 \
  --output_dir runs/victim_seed0 \
  2>&1 | tee logs/main_baselines.log

## Performs model inversion on a pretrained EWC model

In [ ]:
CUDA_VISIBLE_DEVICES=0 python -u main_inv.py \
  --pretrained_model_add runs/victim_seed0/checkpoint.pkl \
  --num_samples 128 \
  --save_dir cifar100_inverted_data_ewc \
  --task_lst 0,1,2,3,4,5,6,7,8 \
  --save_every 1000 \
  --batch_reg \
  --init_acc \
  --n_iters 10000 \
  2>&1 | tee logs/model_inversion.log

## Generates a BrainWash data-poisoning attack against an EWC model

In [ ]:
CUDA_VISIBLE_DEVICES=0 python -u main_brainwash.py \
  --extra_desc detector_baseline \
  --pretrained_model_add runs/victim_seed0/checkpoint.pkl \
  --mode reckless \
  --target_task_for_eval 0 \
  --delta 0.3 \
  --seed 0 \
  --eval_every 10 \
  --distill_folder cifar100_inverted_data_ewc \
  --init_acc \
  --noise_norm inf \
  --cont_learner_lr 0.001 \
  --n_epochs 5000 \
  --n_iters 1 \
  --reset_head_every 1 \
  --save_every 100 \
  --output_dir runs/attack_seed0 \
  2>&1 | tee logs/data_poisoning.log

## Trains task 9 on clean data

In [ ]:
CUDA_VISIBLE_DEVICES=0 python -u main_baselines.py \
    --seed 0 \
    --experiment split_cifar100 \
    --approach ewc \
    --lasttask 9 \
    --tasknum 10 \
    --nepochs 20 \
    --batch-size 16 \
    --lr 0.01 \
    --clip 100.0 \
    --lamb 500000 \
    --checkpoint runs/attack_seed0/noise.pkl \
    --init_acc \
    --output_dir runs/attack_effectiveness/seed0/clean/ \
    2>&1 | tee logs/train_task9_on_clean_data.log

## Trains task 9 on poisoned data

In [ ]:
CUDA_VISIBLE_DEVICES=0 python -u main_baselines.py \
    --seed 0 \
    --experiment split_cifar100 \
    --approach ewc \
    --lasttask 9 \
    --tasknum 10 \
    --nepochs 20 \
    --batch-size 16 \
    --lr 0.01 \
    --clip 100.0 \
    --lamb 500000 \
    --checkpoint runs/attack_seed0/noise.pkl \
    --init_acc \
    --addnoise \
    --output_dir runs/attack_effectiveness/seed0/poison/ \
    2>&1 | tee logs/train_task9_on_poison_data.log